# Task 2: Chunking, Embedding & Vector Store Indexing 
Builds a stratified sample, chunks narratives, embeds them, and indexes into ChromaDB with traceable metadata.

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import time

from src.chunking import get_text_splitter, chunk_dataframe
from src.embedding import load_embedding_model, embed_chunks, build_chroma_store, EMBEDDING_MODEL_NAME

FILTERED_PATH = "../data/processed/filtered_complaints.csv"
VECTOR_STORE_DIR = "../vector_store"
SAMPLE_SIZE = 12_500
RANDOM_SEED = 42


c:\Users\AAM3\Desktop\rag-complaint-chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import src.embedding as emb
print(emb.__file__)
print(dir(emb))

c:\Users\AAM3\Desktop\rag-complaint-chatbot\notebooks\..\src\embedding.py
['EMBEDDING_MODEL_NAME', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'build_chroma_store', 'chromadb', 'embed_chunks', 'load_embedding_model']


## 1. Load the cleaned dataset from Task 1

In [3]:
df = pd.read_csv(FILTERED_PATH)
print(f"Loaded {len(df):,} cleaned complaints")
df["product_category"].value_counts()

Loaded 480,560 cleaned complaints


product_category
Credit Card        189333
Savings Account    155202
Money Transfer      98684
Personal Loan       37341
Name: count, dtype: int64

## 2. Stratified Sampling
 We sample proportionally from each product category so that the sample's category mix matches the full filtered dataset's mix. Note: the original Task 2 instructions mention "five product categories," but only four were defined in Task 1 (Credit Card, Personal Loan, Savings Account, Money Transfer) we proceed with these four, treating "five" as a documentation inconsistency.

Target sample size: 12,500 (midpoint of the 10,000-15,000 range).

In [4]:
def stratified_sample(df: pd.DataFrame, strata_col: str, sample_size: int, seed: int = 42) -> pd.DataFrame:
    """
    Proportionally samples `sample_size` rows from `df`, preserving the
    relative frequency of each value in `strata_col`.
    """
    total = len(df)
    parts = []

    for category, group in df.groupby(strata_col, observed=True):
        frac = len(group) / total
        n = round(frac * sample_size)
        n = min(n, len(group))  # safety: never sample more than available
        parts.append(group.sample(n=n, random_state=seed))

    return pd.concat(parts, ignore_index=True)


sample_df = stratified_sample(df, "product_category", SAMPLE_SIZE, seed=RANDOM_SEED)

print(f"Sample size: {len(sample_df):,}")
print("\nSample distribution:")
print(sample_df["product_category"].value_counts())
print("\nOriginal distribution (for comparison):")
print((df["product_category"].value_counts(normalize=True) * 100).round(1))
print("\nSample distribution (% , should closely match above):")
print((sample_df["product_category"].value_counts(normalize=True) * 100).round(1))

Sample size: 12,500

Sample distribution:
product_category
Credit Card        4925
Savings Account    4037
Money Transfer     2567
Personal Loan       971
Name: count, dtype: int64

Original distribution (for comparison):
product_category
Credit Card        39.4
Savings Account    32.3
Money Transfer     20.5
Personal Loan       7.8
Name: proportion, dtype: float64

Sample distribution (% , should closely match above):
product_category
Credit Card        39.4
Savings Account    32.3
Money Transfer     20.5
Personal Loan       7.8
Name: proportion, dtype: float64


## 3. Text Chunking
We use LangChain's `RecursiveCharacterTextSplitter` with `chunk_size=500` and `chunk_overlap=50` characters. This matches the spec of the pre-built `complaint_embeddings.parquet` store used in Tasks 3-4, keeping chunk granularity consistent across the whole project even though this index is built independently at smaller scale.
Rationale:
- 500 characters is roughly 80-100 words, large enough to preserve context within a single complaint topic, small enough to keep each chunk's embedding focused on one idea rather than blending several.
- 50 characters of overlap (10%) helps avoid losing meaning at chunk boundaries, e.g. a sentence split mid-thought.

In [5]:
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

splitter = get_text_splitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

start = time.time()
chunk_records = chunk_dataframe(
    sample_df,
    text_column="cleaned_narrative",
    splitter=splitter,
    id_column="Complaint ID",
)
print(f"Created {len(chunk_records):,} chunks from {len(sample_df):,} complaints in {time.time()-start:.1f}s")
print(f"Average chunks per complaint: {len(chunk_records) / len(sample_df):.2f}")

Created 23,114 chunks from 12,500 complaints in 1.4s
Average chunks per complaint: 1.85


### 3a. Sanity check a few chunked examples

In [6]:
for r in chunk_records[:3]:
    print(f"Complaint {r['complaint_id']} | chunk {r['chunk_index']}/{r['total_chunks']-1} | {r['product_category']}")
    print(f"  {r['chunk_text'][:150]}...")
    print()

Complaint 11681312 | chunk 0/0 | Credit Card
  information heist account fraudulent belong would like account completely remove three credit bureau credit report...

Complaint 3974403 | chunk 0/0 | Credit Card
  2018 bill go autobill reach company say saw payment try come bank reject never notify rejection give late payment get mortgage mark credit company hel...

Complaint 3947764 | chunk 0/4 | Credit Card
  2020 apply credit card account comenity bank issue credit card handle account relate matter 2020 place call comenity inquire receive phone call accord...



## 4. Embedding Model
We use `sentence-transformers/all-MiniLM-L6-v2`:
- Produces compact 384-dimensional embeddings (~80MB model size), fast enough to run on CPU for both this sample and later full-scale use.
- Well-suited for semantic similarity / retrieval tasks (trained with a contrastive objective on sentence-pair data).
- Matches the embedding model used in the pre-built `complaint_embeddings.parquet` store, so embeddings are directly comparable if the two stores are ever combined or compared.

In [7]:
print(f"Loading embedding model: {EMBEDDING_MODEL_NAME}")
model = load_embedding_model(EMBEDDING_MODEL_NAME)
print("Model loaded. Embedding dimension:", model.get_sentence_embedding_dimension())

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


c:\Users\AAM3\Desktop\rag-complaint-chatbot\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\AAM3\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5132.57it/s]


Model loaded. Embedding dimension: 384


C:\Users\AAM3\AppData\Local\Temp\ipykernel_10208\3552875409.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Model loaded. Embedding dimension:", model.get_sentence_embedding_dimension())


## 5. Generate Embeddings
This is the most compute-intensive step. Expect several minutes on CPU for ~12,500 complaints (roughly 1.0-1.5 chunks per complaint on average).

In [8]:
start = time.time()
chunk_records = embed_chunks(model, chunk_records, batch_size=64)
print(f"\nEmbedded {len(chunk_records):,} chunks in {time.time()-start:.1f}s")

Batches: 100%|██████████| 362/362 [03:52<00:00,  1.56it/s]



Embedded 23,114 chunks in 233.2s


## 6. Build and Persist the Vector Store
Stored in ChromaDB with per-chunk metadata (`complaint_id`, `product_category`, `chunk_index`, `total_chunks`) so every retrieved chunk can be traced back to its original complaint.

In [9]:
start = time.time()
collection = build_chroma_store(
    chunk_records,
    persist_dir=VECTOR_STORE_DIR,
    collection_name="complaints_sample",
)
print(f"Indexed {collection.count():,} chunks into ChromaDB in {time.time()-start:.1f}s")
print(f"Vector store persisted to: {VECTOR_STORE_DIR}")

Indexed 23,114 chunks into ChromaDB in 17.3s
Vector store persisted to: ../vector_store


## 7. Sanity Check: Run a Test Query

In [10]:
test_query = "unauthorized charges on my credit card"
query_embedding = model.encode([test_query], convert_to_numpy=True)[0].tolist()

results = collection.query(query_embeddings=[query_embedding], n_results=3)

for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
    print(f"[{meta['product_category']}] complaint {meta['complaint_id']} (chunk {meta['chunk_index']})")
    print(f"  {doc[:150]}...")
    print()

[Credit Card] complaint 3809499 (chunk 0)
  bank america case bill card unauthorized charge...

[Credit Card] complaint 3644732 (chunk 0)
  unauthorized charge credit card company refuse remove call multiple time dispute charge first say call back additional information dispute resolve ref...

[Savings Account] complaint 5606623 (chunk 0)
  receive two unauthorized charge debit card card set charge go unless fund ive report unauthorized charge bank refuse remove charge even though report ...



## Task 2 Summary

### Data Sampling, Chunking, and Vector Store Construction

A stratified sample of **12,500 complaints** was drawn from the **480,564 cleaned complaints** produced in Task 1, preserving the original product-category distribution. The resulting sample contained:

- **Credit Card:** 4,925 complaints (39.4%)
- **Savings Account:** 4,037 complaints (32.3%)
- **Money Transfer:** 2,567 complaints (20.5%)
- **Personal Loan:** 971 complaints (7.8%)

These proportions exactly match those of the full dataset, ensuring that the sample remains representative while reducing computational requirements for embedding and retrieval.

Complaint narratives were segmented using LangChain's `RecursiveCharacterTextSplitter` with a **chunk size of 500 characters** and **50-character overlap**. This configuration was selected to balance context preservation with embedding specificity while maintaining consistency with the retrieval granularity used in the downstream RAG pipeline.

The chunking process generated **23,114 text chunks** from the 12,500 sampled complaints, corresponding to an average of **1.85 chunks per complaint**.

Chunks were embedded using the **`sentence-transformers/all-MiniLM-L6-v2`** model, a lightweight transformer model (~80 MB) that produces **384-dimensional dense vector embeddings** optimized for semantic similarity search. Embedding generation for all chunks required approximately **233 seconds**.

The resulting embeddings were indexed into a persisted **ChromaDB** collection. Each chunk was stored together with metadata fields including:

- `complaint_id`
- `product_category`
- `chunk_index`
- `total_chunks`

This metadata ensures complete traceability between retrieved chunks and their source complaints, supporting filtering, debugging, evaluation, and explainability throughout the RAG workflow.